In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
master_mem_df = pd.read_csv('../data/processed/master_mem_df.csv')

THE FIRST CLASSIFIER WILL BE A DUMMY CLASSIFIER, ALL IT DOES IS COUNT THE MAJORITY CLASS

In [ ]:
from sklearn.model_selection import train_test_split
X=master_mem_df.drop(columns=['DESYNPUF_ID', 'DISENGAGED'])
Y=master_mem_df['DISENGAGED']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print(f"Train Set: {X_train.shape}")
print(f"Test Set: {X_test.shape}")

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train,y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"Dummy Accuracy: {accuracy_score(y_test, y_pred_dummy)*100: .2f}%")
print(classification_report(y_test, y_pred_dummy))

BUILDING A LOGISTIC REGRESSION MODEL

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

#scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr=lr.predict(X_test_scaled)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

#scaling the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=2000)
lr.fit(X_train, y_train)
y_pred_lr=lr.predict(X_test_scaled)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%")
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f"Random Forest AccuracyScore: {accuracy_score(y_test, y_pred_rf)*100: .2f}%")
print(classification_report(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import roc_auc_score

#get probability score
y_prob_rf = rf.predict_proba(X_test)[:, 1]
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print(f"Random Forest AUC-ROC: {roc_auc_score(y_test, y_prob_rf): .4f}")
print(f"Logistic Regression AUC-ROC: {roc_auc_score(y_test, y_prob_lr): .4f}")

ACCORDING TO THE AUC SCORE OF 0.779, OUR RANDOM FOREST MODEL CAN PREDICT DISENGAGEMENT 77% PERCENT OF THE TIME. WHICH IS DECENT WITH A PRECISION SCORE OF 0.79! NOW LETS SEE WHAT DRIVES THIS RESULT. WITH THE HELP OF SHAP EXPLAINABILITY WE WILL SEE WHAT FEATURES ARE CONTRIBUTING TO THE CLASSIFICATION.

In [ ]:
import shap

# Create explainer
explainer = shap.TreeExplainer(rf)

# Calculate SHAP values on test set sample
X_test_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(X_test_sample)

print(type(shap_values))
print(len(shap_values))